In [ ]:
  # Install PyTorch, Whisper, FFmpeg, and Pandas
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ffmpeg-python openai-whisper pandas


In [ ]:
!pip install fer==22.5.1

no need to re-run this cell , as drive is already mounted


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob


# Change this to your actual folder path in Drive
VIDEO_FOLDER = "/content/drive/MyDrive/Project_videos/"

# Get all video files
video_files = glob.glob(VIDEO_FOLDER + "*.mp4")
video_files = [os.path.basename(v) for v in video_files]

print(f"✅ Found {len(video_files)} videos:")
for v in video_files:
    print(f"  - {v}")

In [ ]:
import whisper
import pandas as pd

import subprocess

# Load Whisper model
model = whisper.load_model("base")

# Create a folder for audio files
os.makedirs("audio_files", exist_ok=True)

# List to store transcription data
transcription_data = []

for video_name in video_files:
    print(f"Processing: {video_name}")

    video_path = os.path.join(VIDEO_FOLDER, video_name)
    audio_file = f"audio_files/{video_name.replace('.mp4', '.wav')}"

    # Extract audio from video
    subprocess.run([
        'ffmpeg', '-y', '-i', video_path,
        '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
        audio_file
    ], check=True, capture_output=True)

    # Transcribe audio
    result = model.transcribe(audio_file)
    transcription_text = result["text"]

    # Store transcription data
    transcription_data.append([video_name, transcription_text])

# Convert list to DataFrame
df = pd.DataFrame(transcription_data, columns=["Video Name", "Transcription"])
print("✅ Transcription done")
df.head()


In [ ]:
# Save DataFrame to an Excel file
df.to_excel("transcriptions1.xlsx", index=False)

print("Transcriptions saved to transcriptions1.xlsx")


In [ ]:
from google.colab import files
files.download("transcriptions1.xlsx")




Text analysis.


In [ ]:
from google.colab import files

# Upload your Excel file
uploaded = files.upload()

# Get the file name of the uploaded file (replace with your file name)
excel_file = list(uploaded.keys())[0]

In [ ]:
import pandas as pd

# Read the Excel file into a DataFrame
df = pd.read_excel(excel_file)

# Check the contents of the DataFrame
df.head()

In [ ]:
emotions_keywords = {

    "happy": [
        # Single words — match easily
        "happy", "joyful", "excited", "grateful", "thankful",
        "blessed", "cheerful", "wonderful", "fantastic", "amazing",
        "proud", "confident", "energetic", "motivated", "inspired",
        "peaceful", "content", "fulfilled", "optimistic", "positive",
        "thrilled", "delighted", "overjoyed", "ecstatic", "gleeful",
        "smiling", "laughing", "enjoying", "loving", "appreciated",
        "refreshed", "vibrant", "lively", "radiant", "alive",
        "free", "light", "warm", "bright", "good",
        "great", "nice", "fun", "lovely", "pleased",
        # Phrases
        "really good", "very good", "pretty good", "feel good",
        "feeling good", "doing well", "going well", "feels great",
        "feel great", "feeling great", "feel amazing", "feel fantastic",
        "feel happy", "feeling happy", "so happy", "very happy",
        "really happy", "truly happy", "genuinely happy",
        "feel positive", "feeling positive", "feel hopeful",
        "feel peaceful", "feel content", "feel fulfilled",
        "feel proud", "feel confident", "feel motivated",
        "feel inspired", "feel energized", "feel refreshed",
        "feel blessed", "feel grateful", "feel thankful",
        "feel lucky", "feel free", "feel alive",
        "feel at peace", "feel at ease", "feel settled",
        "feel balanced", "feel secure", "feel stable",
        "made my day", "best day", "good day", "beautiful day",
        "having fun", "enjoying myself", "loving this",
        "worked out well", "going right", "falling into place",
        "things are working out", "proud of myself",
        "glad this happened", "glad I did this",
        "looking forward", "feel uplifted", "lifted my mood",
        "feel lighter", "feel warm inside", "feel connected",
        "feel close", "feels special", "enjoying every moment",
        "feel good overall", "high spirits", "in a good mood"
    ],

    "sad": [
        # Single words — unique to sad only
        "sad", "unhappy", "upset", "hurt", "heartbroken",
        "miserable", "gloomy", "tearful", "teary", "weeping",
        "crying", "broken", "disappointed", "regretful", "guilty",
        "ashamed", "rejected", "unwanted", "ignored", "lonely",
        "isolated", "left out", "let down", "defeated", "low",
        "down", "blue", "grief", "sorrow", "mourn",
        "aching", "painful", "suffering", "shaken", "disturbed",
        # Phrases — unique to sad
        "feel sad", "feeling sad", "really sad", "so sad",
        "very sad", "feel down", "feeling down", "feel low",
        "feeling low", "feel hurt", "feel broken", "feel lonely",
        "feel rejected", "feel ignored", "feel left out",
        "feel disappointed", "feel let down", "feel defeated",
        "bad day", "rough day", "hard day", "tough day",
        "not okay", "not fine", "not good", "felt bad",
        "feels bad", "feeling bad", "really bad", "so bad",
        "very bad", "pretty bad", "going wrong", "fell apart",
        "fell wrong", "feel wrong", "hard time", "tough time",
        "makes me sad", "makes me cry", "breaks my heart",
        "miss them", "miss it", "wish things were different"
    ],

    "depressed": [
        # Single words — unique to depressed only
        "depressed", "hopeless", "helpless", "worthless",
        "useless", "meaningless", "pointless", "numb",
        "hollow", "empty", "blank", "withdrawn", "detached",
        "disconnected", "shattered", "trapped", "stuck",
        "drained", "exhausted", "powerless", "directionless",
        # Phrases — unique to depressed
        "feel depressed", "feeling depressed", "feel hopeless",
        "feel helpless", "feel worthless", "feel useless",
        "feel meaningless", "feel pointless", "feel empty",
        "feel numb", "feel hollow", "feel blank", "feel trapped",
        "feel stuck", "feel lost", "feel disconnected",
        "feel detached", "feel withdrawn", "no energy",
        "no motivation", "lost motivation", "lost interest",
        "no interest", "no feelings", "feel nothing",
        "no joy", "no happiness", "nothing feels good",
        "can't feel happy", "hard to feel anything",
        "mentally tired", "emotionally drained", "completely drained",
        "burnt out", "burned out", "no point", "what's the point",
        "going nowhere", "nothing ahead", "no hope", "zero hope",
        "given up", "want to give up", "done with everything",
        "just done", "so done", "can't anymore",
        "dark thoughts", "dark place", "in darkness",
        "rock bottom", "lowest point", "really low",
        "dead inside", "broken inside", "weak inside",
        "falling apart", "breaking down", "torn apart",
        "heavy inside", "feels heavy", "sinking feeling",
        "deep sadness", "always tired", "zoned out", "checked out"
    ],

    "anxious": [
        # Single words — unique to anxious only
        "anxious", "worried", "stressed", "tense", "uneasy",
        "panicky", "fearful", "scared", "afraid", "concerned",
        "overthinking", "unsettled", "shaky", "trembling",
        "breathless", "sweaty", "jittery", "fidgety", "dizzy",
        "lightheaded", "confused", "uncertain", "doubtful",
        "hesitant", "indecisive", "vulnerable", "insecure",
        "unstable", "pressure",
        # Phrases — unique to anxious only
        "feel anxious", "feeling anxious", "feel worried",
        "feeling worried", "feel stressed", "feeling stressed",
        "feel tense", "feel uneasy", "feel scared", "feel afraid",
        "feel fearful", "feel concerned", "feel uncertain",
        "feel insecure", "feel vulnerable", "feel unstable",
        "on edge", "panicking", "panic attack",
        "overthinking everything", "thinking too much",
        "can't relax", "hard to relax", "can't focus",
        "losing focus", "mind racing", "racing thoughts",
        "thoughts spinning", "can't think straight",
        "blank mind", "foggy mind", "mental fog",
        "heart racing", "fast heartbeat", "heavy breathing",
        "short of breath", "tight chest", "chest tight",
        "head spinning", "cold sweat", "under pressure",
        "feeling pressure", "stressed out", "so stressed",
        "really stressed", "very stressed", "second guessing",
        "self doubt", "no control", "losing control",
        "out of control", "unsafe feeling", "uneasy feeling",
        "something wrong", "feels off", "gut feeling wrong"
    ],

    "nervous": [
        # Single words — unique to nervous only
        "nervous", "apprehensive", "timid", "shy", "awkward",
        "self conscious", "underconfident", "queasy",
        # Phrases — unique to nervous only
        "feel nervous", "feeling nervous", "very nervous",
        "so nervous", "really nervous", "a bit nervous",
        "slightly nervous", "kinda nervous", "pretty nervous",
        "hands shaking", "legs shaking", "voice shaking",
        "cold hands", "sweaty palms", "clammy hands",
        "pounding heart", "stomach knots", "tight stomach",
        "butterflies in stomach", "lost words", "forgetting words",
        "stumbling words", "mixed up words",
        "out of place", "low confidence", "no confidence",
        "feel awkward", "feeling awkward", "feel shy",
        "feel self conscious", "feel apprehensive",
        "blank mind before speaking", "freeze up",
        "can't speak", "tongue tied", "at a loss for words",
        "too much pressure talking", "panic before speaking",
        "mini panic", "inner tension before talking"
    ],

    "sarcasm": [
        # Only keep phrases that are CLEARLY sarcastic in context
        # Removed all genuinely positive single words
        "yeah right", "sure thing", "oh sure",
        "oh obviously", "oh clearly", "oh totally",
        "what a surprise", "big surprise", "shocking right",
        "who knew", "you don't say", "oh really now",
        "good for you", "must be nice", "must be great for you",
        "yeah sure why not", "makes total sense",
        "seems legit", "sounds perfect to you",
        "as if", "like really", "okay then sure",
        "oh fine", "just great", "just wow",
        "nice going", "smooth move there",
        "well played", "genius move", "very smart of you",
        "so impressive", "real smart", "nice work there",
        "be my guest", "don't mind me",
        "sure go ahead", "yeah okay sure",
        "sure whatever", "makes it better somehow",
        "even better right", "couldn't be better huh",
        "just what I needed right", "exactly what I wanted",
        "perfect plan", "nothing wrong at all",
        "all perfect", "best idea ever",
        "oh how lovely", "how very nice of you",
        "great timing as always", "perfect timing right"
    ]
}

In [ ]:
import re

def classify_emotion(text):
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)

    scores = {
        "happy": 0,
        "sad": 0,
        "anxious": 0,
        "depressed": 0,
        "sarcasm": 0,
        "nervous": 0
    }

    for emotion, keywords in emotions_keywords.items():
        for keyword in keywords:
            if " " in keyword:
                if keyword in text:
                    scores[emotion] += 1
            else:
                if keyword in words:
                    scores[emotion] += 1

    negative_score = (
        scores["sad"] +
        scores["anxious"] +
        scores["depressed"] +
        scores["nervous"]
    )
    positive_score = scores["happy"]
    total_score = negative_score + positive_score

    # 🔥 Sarcasm detection — happy words + strong negative context
    if positive_score > 0 and negative_score > 0:
        if negative_score >= 2:
            return "sarcasm", -2

    # 🔥 Ratio based logic
    if total_score > 0:
        positive_ratio = positive_score / total_score

        # Happy wins if it dominates (60%+ of all matches)
        if positive_ratio >= 0.6:
            return "happy", 2

    # 🔥 Negative priority (only when happy is weak or absent)
    if scores["depressed"] > 0:
        return "depressed", -4
    elif scores["anxious"] > 0:
        return "anxious", -3
    elif scores["sad"] > 0:
        return "sad", -2
    elif scores["happy"] > 0:
        return "happy", 2
    else:
        return "neutral", 0

In [ ]:
# Apply emotion classification to each transcription in the 'Transcription' column
df[["Emotion", "Emotion_Score"]] = df["Transcription"].apply(
    lambda x: pd.Series(classify_emotion(x))
)
# Show the result
df.head()


In [ ]:
# Save the updated DataFrame to an Excel file
df.to_excel("transcriptions_with_emotions.xlsx", index=False)

# Download the updated Excel file
from google.colab import files
files.download("transcriptions_with_emotions.xlsx")


In [ ]:
# After runtime restarts, re-run your original imports first, then:
import cv2
from fer import FER
from collections import Counter
print("✅ FER imported successfully")

In [ ]:
detector = FER(mtcnn=False)
print("✅ FER detector ready")

In [ ]:
facial_score_map = {
    "angry": -3,
    "disgust": -3,
    "fear": -3,
    "sad": -2,
    "neutral": 0,
    "surprise": 1,
    "happy": 2
}

def analyze_video_emotions(video_path, frame_skip=15):
    cap = cv2.VideoCapture(video_path)
    emotions_found = []
    frame_count = 0

    if not cap.isOpened():
        print(f"  ❌ Cannot open: {video_path}")
        return "neutral", 0, {}

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_skip == 0:
            try:
                frame_resized = cv2.resize(frame, (640, 480))
                result = detector.detect_emotions(frame_resized)

                if result:
                    scores = result[0]['emotions']
                    dominant = max(scores, key=scores.get)
                    if scores[dominant] > 0.4:
                        emotions_found.append(dominant)

            except Exception:
                pass

        frame_count += 1

    cap.release()

    if not emotions_found:
        return "neutral", 0, {}

    emotion_counts = Counter(emotions_found)
    dominant_emotion = emotion_counts.most_common(1)[0][0]
    total = len(emotions_found)
    breakdown = {k: round(v / total * 100, 1) for k, v in emotion_counts.items()}

    # Get score for dominant emotion
    emotion_score = facial_score_map.get(dominant_emotion, 0)

    return dominant_emotion, emotion_score, breakdown

In [ ]:
print("🎬 Starting facial analysis on all videos...\n")

facial_emotions = []
facial_scores = []
facial_breakdowns = []

for video_name in video_files:
    print(f"  Processing: {video_name}")

    video_path = os.path.join(VIDEO_FOLDER, video_name)  # ← full path
    dominant, score, breakdown = analyze_video_emotions(video_path)  # ← pass video_path not video_name

    facial_emotions.append(dominant)
    facial_scores.append(score)
    facial_breakdowns.append(str(breakdown))
    print(f"  ✅ Dominant: {dominant} | Score: {score} | Breakdown: {breakdown}\n")

df["Facial_Emotion"] = facial_emotions
df["Facial_Emotion_Score"] = facial_scores
df["Facial_Emotion_Breakdown"] = facial_breakdowns

In [ ]:
output_file = "student_emotion_analysis.xlsx"
df.to_excel(output_file, index=False)
files.download(output_file)
print("✅ Saved and downloaded!")

audio analysis

In [ ]:
!pip install -q librosa==0.9.2 soundfile

In [ ]:
import librosa
import numpy as np
import soundfile
print("✅ Audio imports ready")

In [ ]:
def extract_audio_features(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000)

        if np.mean(np.abs(y)) < 0.001:
            print(f"  ⚠️ Skipping {audio_path} — too silent")
            return None

        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        rms = np.mean(librosa.feature.rms(y=y))
        pitch, _ = librosa.piptrack(y=y, sr=sr)
        pitch = pitch[pitch > 0]
        avg_pitch = np.mean(pitch) if len(pitch) > 0 else 0

        return list(mfccs) + [zcr, rms, avg_pitch]

    except Exception as e:
        print(f"  ❌ Error processing {audio_path}: {e}")
        return None

print("✅ extract_audio_features ready")

In [ ]:
os.makedirs("audio_files", exist_ok=True)
audio_features = []

for video_name in video_files:
    print(f"  Extracting: {video_name}")
    audio_file = f"audio_files/{video_name.replace('.mp4', '.wav')}"

    if not os.path.exists(audio_file):
        subprocess.run([
            'ffmpeg', '-y', '-i', video_name,
            '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
            audio_file
        ], check=True, capture_output=True)

    features = extract_audio_features(audio_file)
    if features:
        audio_features.append([video_name] + features)

mfcc_cols = [f"MFCC_{i+1}" for i in range(13)]
columns = ["Video Name"] + mfcc_cols + ["ZCR", "RMS", "Avg_Pitch"]
audio_df = pd.DataFrame(audio_features, columns=columns)

print(f"✅ Audio features extracted for {len(audio_df)} videos")
audio_df.head()

In [ ]:
# Merge audio features into main df
df = pd.merge(df, audio_df, on="Video Name", how="left")

# Assign conservative label
def get_conservative_label(row):
    text_score = row["Emotion_Score"]
    facial_score = row["Facial_Emotion_Score"]
    if text_score <= facial_score:
        return row["Emotion"], text_score
    else:
        return row["Facial_Emotion"], facial_score

df[["Audio_Label", "Audio_Label_Score"]] = df.apply(
    lambda row: pd.Series(get_conservative_label(row)), axis=1
)

print("✅ Labels assigned")
print(df[["Video Name", "Emotion", "Emotion_Score",
          "Facial_Emotion", "Facial_Emotion_Score",
          "Audio_Label", "Audio_Label_Score"]])

In [ ]:
df.to_excel("complete_dataset.xlsx", index=False)
files.download("complete_dataset.xlsx")
print("✅ Complete dataset saved!")